In [2]:
# dcdc abstraction with PAC guarantee example

# needed libraries
import numpy as np
import scipy.special as sp
import time
from itertools import product
import random
import gurobipy as gp
from gurobipy import GRB
from joblib import Parallel, delayed
from scipy.optimize import fsolve
from math import comb
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import scipy.stats as stats
from scipy.optimize import brentq
from scipy.stats import truncnorm

In [28]:
# choice of N for repetitve scenario
N = 1000
n_dim = 2 # dimension of state set
N_pos = 100 # N used for computing post in abstraction

In [29]:
# system dynamics
tau = .8 #sampling time
eta_x1 = 0.0005 
eta_x = np.array([eta_x1,eta_x1]) # discretization vector
# set of states
a1, a2, b1, b2 = 0,1.55,5.45,5.85
a = np.array([a1, b1])  # lower bounds
b = np.array([a2, b2])  # upper bounds

dim1_hat = np.arange(a1+eta_x1, a2-eta_x1, eta_x1)
dim2_hat = np.arange(b1+eta_x1, b2-eta_x1, eta_x1)
X_ha = np.array(list(product(dim1_hat, dim2_hat))) 
X_haa = np.around(X_ha, 2)

In [30]:
# # Sample N i.i.d. pairs (x_i, x_hat_i)
# x_samples = np.random.uniform(low=a, high=b, size=(N, 2))
# x_hat_indices = np.random.choice(len(X_haa), size=N, replace=True)
# x_hat_samples = X_haa[x_hat_indices]

# # Combine into a list of tuples or array of pairs
# X_pairs = list(zip(x_samples, x_hat_samples))
# len(X_pairs)

In [31]:
# # --- Truncated normal for continuous x_samples ---
# def sample_truncnorm(a, b, mean, std, size):
#     a_, b_ = (a - mean)/std, (b - mean)/std
#     return truncnorm.rvs(a_, b_, loc=mean, scale=std, size=size)

# # Example parameters
# mean = (a + b)/2
# std = (b - a)/4  # roughly cover the interval

# x_samples = sample_truncnorm(a, b, mean, std, size=(N, 2))

# # --- Gaussian sampling for x_hat_samples from discrete grid X_haa ---
# def sample_from_grid_gaussian(X_grid, N, mean_point=0, std_dev=np.sqrt(0.1)):
#     """
#     X_grid: array of shape (M, dim) representing the discrete grid
#     mean_point: point in grid coordinates (index) to center Gaussian around
#     std_dev: controls spread over grid indices
#     """
#     M = len(X_grid)
#     if mean_point is None:
#         mean_point = M / 2
#     # Truncated normal over indices
#     a_, b_ = (0 - mean_point)/std_dev, (M-1 - mean_point)/std_dev
#     idx = truncnorm.rvs(a_, b_, loc=mean_point, scale=std_dev, size=N).astype(int)
#     return X_grid[idx]

# x_hat_samples = sample_from_grid_gaussian(X_haa, N, mean_point=len(X_haa)/2, std_dev=len(X_haa)/6)

# # --- Combine into pairs ---
# X_pairs = list(zip(x_samples, x_hat_samples))
# print(len(X_pairs))

In [32]:
# --- Uniform sampling for continuous x_samples ---
def sample_uniform(a, b, size):
    """
    Sample uniformly in [a, b] for each dimension.
    """
    return np.random.uniform(low=a, high=b, size=size)

# Example parameters
# a, b, N must be defined externally
x_samples = sample_uniform(a, b, size=(N, 2))


# --- Uniform sampling for x_hat_samples from discrete grid X_grid ---
def sample_from_grid_uniform(X_grid, N):
    """
    X_grid: array of shape (M, dim) representing the discrete grid
    Uniformly sample N points from the grid.
    """
    M = len(X_grid)
    idx = np.random.randint(0, M, size=N)
    return X_grid[idx]

x_hat_samples = sample_from_grid_uniform(X_haa, N)


# --- Combine into pairs ---
X_pairs = list(zip(x_samples, x_hat_samples))
print(len(X_pairs))

1000


In [33]:
# system dynamics as black-box simulator
# dynamics parameters 
r0 = 1.0  
vs = 1.0 
rl = 0.05 
rc = rl / 10 
xl = 3.0 
xc = 70.0 
xd = 20.0

def sys_dyn(x, u):
    b=[vs/xl + np.abs(x[0])**0.5/xd, 0]
    a = np.zeros((2, 2))
    if u == 1: 
      a[0][0] = -rl / xl
      a[0][1] = 0
      a[1][0] = 0
      a[1][1] = (-1 / xc) * (1 / (r0 + rc))
    else:
      a[0][0] = (-1 / xl) * (rl + ((r0 * rc) / (r0 + rc)))
      a[0][1] = ((-1 / xl) * (r0 / (r0 + rc))) / 5 
      a[1][0] = 5 * (r0 / (r0 + rc)) * (1 / xc)
      a[1][1] = (-1 / xc) * (1 / (r0 + rc)) 
    
    f_xt1 = x[0] + tau*(a[0][0]*x[0]+a[0][1]*x[1] + b[0])
    f_xt2 = x[1] + tau*(a[1][0]*x[0]+a[1][1]*x[1] + b[1])
    nxt = [max(min(f_xt1, a2), a1), max(min(f_xt2, b2), b1)]
    # nxt = [f_xt1, f_xt2]
    nxt = [round(x, 2) for x in nxt]
    return nxt

def generate_grid_centers(a, b, N_pos):
    dim = len(a)
    # Estimate number of points per dimension (evenly)
    k = int(np.round(N_pos ** (1 / dim)))
    total_points = k ** dim

    # Generate grid centers per dimension
    grid_axes = []
    for ai, bi in zip(a, b):
        step = (bi - ai) / k
        centers = ai + (np.arange(k) + 0.5) * step
        grid_axes.append(centers)

    # Cartesian product of all centers (i.e., subgrid centers)
    all_points = np.array(list(product(*grid_axes)))

    # Trim if more than needed
    if total_points > N_pos:
        all_points = all_points[:N_pos]

    return all_points

# for abstract system
def sys_dyn_hat(y, u):
    y = np.asarray(y)
    X_hat_arra = np.array(X_haa)  
    a, b = y-eta_x/2, y+eta_x/2 
    # sampling N points as subgrid centres for each cell to obtain an estimate of the reachable sets as done in the paper
    sampled_points = generate_grid_centers(a, b, N_pos)
    # sampling N iid points from each cell to obtain an estimate of the reachable sets
    # sampled_points = np.random.uniform(a, b, size=(N_pos, len(a)))

    # Compute successors
    successors = np.array([sys_dyn(x, u) for x in sampled_points])

    # Compute mean and max distance
    m = np.mean(successors, axis=0)
    r = np.max(np.abs(successors - m), axis=0)

    # Find points in X_hat_array within the under-approximation of reachable sets
    mask = np.all(np.abs(X_hat_arra - m) <= (r + eta_x / 2), axis=1)

    return X_hat_arra[mask]

In [29]:
def lipschitz_bound(x, u):
    """
    Computes a local Lipschitz bound ||J(x)|| for the dynamics.

    Parameters
    ----------
    x : array-like, shape (2,)
    u : int
        Mode (1 or otherwise)
    eps : float
        Small regularization to avoid division by zero near x[0]=0

    Returns
    -------
    L : float
        Local Lipschitz bound
    """

    x0 = x[0]
    tau=0.8
    # system matrix A
    A = np.zeros((2, 2))

    if u == 1:
        A[0, 0] = tau*(-rl / xl) +1
        A[0, 1] = 0
        A[1, 0] = 0
        A[1, 1] = tau*((-1 / xc) * (1 / (r0 + rc))) +1

    else:
        A[0, 0] = tau*((-1 / xl) * (rl + ((r0 * rc) / (r0 + rc)))) +1
        A[0, 1] = tau*(((-1 / xl) * (r0 / (r0 + rc))) / 5)
        A[1, 0] = tau*(5 * (r0 / (r0 + rc)) * (1 / xc))
        A[1, 1] = tau*((-1 / xc) * (1 / (r0 + rc))) +1

    # derivative of nonlinear term
    dphi = 0.0625*tau*1.0 / (2.0 * np.sqrt(np.abs(x0)))

    # Jacobian
    J = A.copy()

    # Lipschitz bound (spectral norm)
    L = np.linalg.norm(J, 2)
    # L = np.linalg.norm(J, np.inf)
    L += dphi

    return L

print("when u=1", lipschitz_bound([5e-10,5.75],1))
print("when u=2", lipschitz_bound([5e-10,5.75],2))

when u=1 1119.0226170370306
when u=2 1119.0250111502396


In [34]:
# set of inputs
U = [1, 2]
M = len(U)
U_array = np.array(U)
print(M)

ubpr = 10
lbp, ubp = -100, 100
lbpc, ubpc = -100, 100
lbe, ube = -10, 0 

2


#### the SOP

In [35]:
m1 = gp.Model("PAC_ASF")

vartheta = m1.addVar(vtype = GRB.CONTINUOUS, name = "vartheta", lb = lbe, ub = ube)

eps_vars = 1/len(X_pairs) 
# eps_vars = {l:m1.addVar(vtype = GRB.CONTINUOUS, name = f"eps_{l}", lb = 0, ub = ubpr) for l in range(len(X_pairs))}
# sum_eps = gp.quicksum(eps_var for eps_var in eps_vars)
# m1.addConstr(sum_eps == 1)

# eps_varsp = {l:m1.addVar(vtype = GRB.CONTINUOUS, name = f"epsp_{l}", lb = 0, ub = ubpr) for l in range(len(X_pairs))}
# sum_epsp = gp.quicksum(eps_var for eps_var in eps_varsp)
# m1.addConstr(sum_epsp == 1)
m1.update()

In [36]:
gamma_b = 1e4 
delta_b = gamma_b * np.linalg.norm(np.array(eta_x), ord=np.inf) **2 # ||eta_x||^2 * gamma
# gamma_b = m1.addVar(vtype = GRB.CONTINUOUS, name = "gamma_b", lb = 10, ub = ubpr)
# delta_b = gamma_b * np.linalg.norm(np.array(eta_x), ord=np.inf) **2 # ||eta_x||^2 * gamma
tau_b = 1.501 #m1.addVar(vtype = GRB.CONTINUOUS, name = "tau_b", lb = 0, ub = ubpr) # this is \rho in the paper
print("delta: ", delta_b)
print("epsilon: ", (delta_b/gamma_b)**0.5)

num_variables = 15 #32  # number of coefficients as a result of the chosen degree of asf
variable_names = [f"lambda_{i}" for i in range(1, num_variables + 1)]
variable_objs = {name_i: m1.addVar(vtype=GRB.CONTINUOUS, name=name_i, lb=lbpc, ub=ubpc) for name_i in variable_names}

# def Asf(x, x_hat):
#     # Coefficients from the declared variables
#     c1,c2,c3,c4,c5 = variable_objs.values()
#     x1, x2, y1, y2 = x[0], x[1], x_hat[0], x_hat[1]
#     # Polynomial expression using the variables
#     result = (
#         c1 + c2*x1 + c3*x2 + c4*y1 + c5*y2
#     )
#     return result

def Asf(x, x_hat):
    # Coefficients from the declared variables
    c1,c2,c3,c4,c5,c6,c7,c8,c9,c10,c11,c12,c13,c14,c15 = variable_objs.values()
    x1, x2, y1, y2 = x[0], x[1], x_hat[0], x_hat[1]
    # Polynomial expression using the variables
    result = (
        c1 + c2*x1 + c3*x2 + c4*y1 + c5*y2 +
        c6*x1**2 + c7*x1*x2 + c8*x1*y1 + c9*x1*y2 +
        c10*x2**2 + c11*x2*y1 + c12*x2*y2 +
        c13*y1**2 + c14*y1*y2 + 
        c15*y2**2
    )
    return result

# def Asf(x, x_hat):
#     # Coefficients from the declared variables
#     c1,c2,c3,c4,c5,c6,c7,c8,c9,c10,c11,c12,c13,c14,c15,c16,c17,c18,c19,c20,c21,c22,c23,c24,c25,c26,c27,c28,c29,c30,c31,c32 = variable_objs.values()
#     x1, x2, y1, y2 = x[0], x[1], x_hat[0], x_hat[1]
#     # Polynomial expression using the variables
#     result = (
#         c1
#         + c2*x1 + c3*x2 + c4*y1 + c5*y2
#         + c6*x1**2 + c7*x1*x2 + c8*x1*y1 + c9*x1*y2
#         + c10*x2**2 + c11*x2*y1 + c12*x2*y2
#         + c13*y1**2 + c14*y1*y2 + c15*y2**2
#         + c16*x1**3 + c17*x1**2*x2 + c18*x1**2*y1 + c19*x1**2*y2
#         + c20*x1*x2**2 + c21*x1*y1**2 + c22*x1*y2**2
#         + c23*x2**3 + c24*x2**2*y1 + c25*x2**2*y2
#         + c26*y1**3 + c27*y1**2*y2 + c28*y2**3
#         + c29*x1*x2*y1 + c30*x1*x2*y2 + c31*x1*y1*y2 + c32*x2*y1*y2
#     )
#     return result

m1.update()

delta:  0.0025
epsilon:  0.0005


In [37]:
# Define lambda functions for efficiency
max_abs_diff = lambda x, y: gamma_b * np.max(np.abs(x - y))**2
asf_value = lambda x, y: Asf(x, y)

# First ASF condition
t_init = time.time()
# sum1 = gp.quicksum(eps_vars[l] * (max_abs_diff(x, y) - asf_value(x, y)) for l, (x,y) in enumerate(X_pairs))
sum1 = gp.quicksum(eps_vars * (max_abs_diff(x, y) - asf_value(x, y)) for (x,y) in X_pairs)
m1.addConstr(sum1 <= vartheta)
t_fin = time.time()
diff_t1 = t_fin - t_init
print(diff_t1)
m1.update()

0.06659722328186035


In [38]:
t_init = time.time()
# ASF 2 Enforce sum2 - Asf <= vartheta

X0 = [X_pairs[l][0] for l in range(len(X_pairs))]
X1 = [X_pairs[l][1] for l in range(len(X_pairs))]
U_list = list(U)

# Only parallelize the pure numerical dynamics (no Gurobi objects)
def compute_dynamics(x0, x1, u):
    dyn_hat_out = sys_dyn_hat(x1, u)  
    dyn_out     = sys_dyn(x0, u)      
    return dyn_out, list(dyn_hat_out)

dyn_results = Parallel(n_jobs=-1, backend="threading", verbose=2)(
    delayed(compute_dynamics)(X0[l], X1[l], U_list[i])
    for l in range(len(X_pairs))
    for i in range(len(U_list))
)

# All Gurobi work (asf_value, addConstr) stays sequential in main process
m1.addConstrs(
    (
        gp.quicksum(
            asf_value(dyn_results[l * len(U_list) + i][0], yn) / len(dyn_results[l * len(U_list) + i][1]) # sigma taken uniformly
            for yn in dyn_results[l * len(U_list) + i][1]
        )
        - tau_b * asf_value(X0[l], X1[l]) + (tau_b - 1) * delta_b <= vartheta
        for l in range(len(X_pairs))
        for i in range(len(U_list))
    ),
    name="asf2"
)

m1.update()
t_fin = time.time()
print(f"Setup time: {t_fin - t_init:.2f}s")

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=-1)]: Done  21 tasks      | elapsed:    0.2s
[Parallel(n_jobs=-1)]: Done 142 tasks      | elapsed:    1.1s
[Parallel(n_jobs=-1)]: Done 345 tasks      | elapsed:    2.6s
[Parallel(n_jobs=-1)]: Done 628 tasks      | elapsed:    4.7s
[Parallel(n_jobs=-1)]: Done 993 tasks      | elapsed:    7.4s
[Parallel(n_jobs=-1)]: Done 1438 tasks      | elapsed:   10.6s
[Parallel(n_jobs=-1)]: Done 1965 tasks      | elapsed:   14.4s
[Parallel(n_jobs=-1)]: Done 2000 out of 2000 | elapsed:   14.7s finished


Setup time: 32.65s


In [39]:
m1.Params.LogToConsole = 0
m1.setParam("NumericFocus", 3)
m1.setParam('NonConvex', 2)

m1.update()
print("Start now")
print("Number of variables:", m1.numVars)
print("Number of constraints:", m1.numConstrs)
t_init = time.time()

m1.setObjective(vartheta, GRB.MINIMIZE)
m1.setParam('Presolve', 0)

m1.optimize()
t_fin = time.time()
diff_t3 = t_fin - t_init
print("opt_status:", m1.status)

m1.write("dcdc_with_PAC_Infeasible_most_recent.lp")

# Check if optimization was successful
if m1.status == GRB.OPTIMAL:
    with open('dcdc_with_PAC_variables_most_recent.txt', 'w') as file:
        for v in m1.getVars():
            file.write(f'{v.VarName} {v.X}\n')
elif m1.status == GRB.INFEASIBLE:
    print("Model is infeasible. Computing IIS...")
    m1.computeIIS()
    m1.write("dcdc_with_PAC_Infeasible_most_recent.ilp")
    print("IIS written to dcdc_with_PAC_Infeasible_most_recent.ilp")

    with open("dcdc_with_PAC_Infeasible_most_recent.txt", "w") as f:
        for c in m1.getConstrs():
            if c.IISConstr:
                f.write(f"Infeasible constraint: {c.ConstrName}\n")
        for v in m1.getVars():
            if v.IISLB or v.IISUB:
                f.write(f"Infeasible bound on variable: {v.VarName}\n")

else:
    print("Optimization was not successful (not optimal or infeasible). Status:", m1.status)

# Count binding constraints
sup_const = [constr for constr in m1.getConstrs() if abs(constr.Slack) < 1e-4]
s_ = len(sup_const)
print(f"Number of constraints within tol: {s_}")
# print("Time (in s):", diff_t1+diff_t2+diff_t3)
print("Time (in s):", diff_t3)

Start now
Number of variables: 16
Number of constraints: 2001
opt_status: 2
Number of constraints within tol: 0
Time (in s): 0.001683950424194336


In [27]:
# def prune_constraints_inf_norm(m1, tol=1e-4, n_jobs=-1):
#     # Solve original model once
#     m1.optimize()
#     if m1.status != gp.GRB.OPTIMAL:
#         raise RuntimeError("Original model is not optimal.")

#     sol_star = np.array([v.X for v in m1.getVars()])

#     # --- Free pre-filter: skip constraints with large slack (clearly non-binding) ---
#     C = list(m1.getConstrs())
#     candidates = []
#     for constr in C:
#         slack = constr.Slack
#         # Only test constraints that are near-active (slack ≈ 0)
#         if abs(slack) <= tol * 10:
#             candidates.append(constr)
    
#     print(f"Total constraints: {len(C)}, candidates to test: {len(candidates)}")

#     # --- Parallel testing of candidate constraints ---
#     # We pass the model as a file to avoid pickling Gurobi objects
#     import tempfile, os
#     with tempfile.NamedTemporaryFile(suffix=".mps", delete=False) as f:
#         tmp_path = f.name
#     m1.write(tmp_path)

#     # Get constraint names for candidates
#     candidate_names = [c.ConstrName for c in candidates]
#     sol_star_copy = sol_star.copy()

#     def test_constraint(constr_name):
#         import gurobipy as gp_local
#         import numpy as np_local

#         env = gp_local.Env()
#         env.setParam("OutputFlag", 0)  # suppress output
#         env.setParam("LogToConsole", 0)

#         m_copy = gp_local.read(tmp_path, env)
#         m_copy.setParam("OutputFlag", 0)

#         c = m_copy.getConstrByName(constr_name)
#         if c is None:
#             return constr_name, False
#         m_copy.remove(c)
#         m_copy.update()
#         m_copy.optimize()

#         if m_copy.status == gp_local.GRB.OPTIMAL:
#             sol_new = np_local.array([v.X for v in m_copy.getVars()])
#             diff = np_local.linalg.norm(sol_star_copy - sol_new, ord=np_local.inf)
#             return constr_name, diff > tol
#         return constr_name, False  # infeasible/unbounded → not necessary per your logic

#     results = Parallel(n_jobs=n_jobs, backend="loky", verbose=2)(
#         delayed(test_constraint)(name) for name in candidate_names
#     )

#     os.unlink(tmp_path)  # clean up temp file

#     # Collect necessary constraints
#     necessary_names = {name for name, is_necessary in results if is_necessary}
#     necessary_constraints = [c for c in C if c.ConstrName in necessary_names]

#     return necessary_constraints

# # Usage
# s_N = len(prune_constraints_inf_norm(m1, tol=1e-4))
# print(f"Number of support constraints: {s_N}")

Total constraints: 4001, candidates to test: 0
Number of support constraints: 0


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=-1)]: Done   0 out of   0 | elapsed:    0.0s finished


In [12]:
# Count binding constraints
def prune_constraints_inf_norm(m1, tol=1e-4):
    """
    Iteratively test which constraints are necessary
    based on their effect on the optimal solution.
    Uses infinity norm for solution difference.
    Returns the final set of constraints deemed necessary.
    """
    # Solve original model
    m1.optimize()
    if m1.status != gp.GRB.OPTIMAL:
        raise RuntimeError("Original model is not optimal.")
    
    # Extract original solution sol*
    sol_star = np.array([v.X for v in m1.getVars()])
    
    # Initial set of constraints
    C = list(m1.getConstrs())
    
    necessary_constraints = []
    
    for constr in C:
        # Create a clone of the model
        m_copy = m1.copy()
        
        # Remove the constraint from the copy
        constr_copy = m_copy.getConstrByName(constr.ConstrName)
        m_copy.remove(constr_copy)
        m_copy.update()
        
        # Solve the reduced model
        m_copy.optimize()
        
        if m_copy.status == gp.GRB.OPTIMAL:
            sol_starstar = np.array([v.X for v in m_copy.getVars()])
            diff = np.linalg.norm(sol_star - sol_starstar, ord=np.inf)  # infinity norm
            
            # If the solution shifts a lot, keep the constraint
            if diff > tol:
                necessary_constraints.append(constr)
        # else:
        #     # If infeasible or unbounded without this constraint, it’s definitely necessary
        #     necessary_constraints.append(constr)
    
    return necessary_constraints

# Usage
s_N = len(prune_constraints_inf_norm(m1, tol=1e-4))
print(f"Number of support constraints: {s_N}")

Number of support constraints: 1


#### nonconvex SOP PAC bound
Consider Equation (7) in https://ieeexplore.ieee.org/stamp/stamp.jsp?tp=&arnumber=8299432

#### comparison of $\beta,\alpha$, and $\mathcal{N}$.

In [13]:
# as alternative, consider eqn 7 in the paper
def epsil(k,N1,beta1):
    res = (beta1/(N1*comb(N1, k)))**(1/(N1-k))
    return 1-res

In [16]:
beta = 10**-6
alpha_sN = epsil(4, N, beta)#epsil(s_N, N, beta)

print(f"With confidence of {(1-beta)*100:.6f}%, the non-violation of at least {1-alpha_sN:.6f}, and violation {alpha_sN:.6f}")

With confidence of 99.999900%, the non-violation of at least 0.955661, and violation 0.044339
